# Khảo Sát Tích Tụ Nhãn OCR Từ Tập Dữ Liệu TextOCR (Phase 0.7 TextOCR Explore)

Tuyển tập phân tích này đi sâu vào mổ xẻ đặc tính tham số học của các đoạn nhãn văn bản tích hợp bối cảnh (scene-text annotations) trong bộ dữ liệu TextOCR. Do khối lượng dữ liệu khổng lồ, việc tối ưu hóa quản lý không gian thông tin được thiết lập.
Bảo chứng mô hình (Fail-safe Guarantee): Toàn bộ logic giải mã đều áp dụng cơ chế đánh bắt ngoại lệ rỗng (Null-exception catching) để không đứt quãng luồng làm việc nếu dữ liệu thô (raw datasets) chưa được định danh tại `data/external`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.training.scripts.data_utils import load_yaml

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (10, 6), "axes.titlesize": 14})

repo = Path("..").resolve()
config = load_yaml(repo / "configs/datasets/ocr_textocr.yaml")
normalized_path = repo / config["output"]["normalized_annotations"]

print(f"Định tuyến dữ liệu chuẩn hóa OCR: {normalized_path}")

## 1. Duyệt Mã Tuần Tự (Sequential Iteration Pipeline)
Việc đọc chuỗi JSON lớn nguyên khối sẽ gây ra hiệu ứng dội ngược về bộ nhớ đệm (Out-of-memory overflow). 
Cơ chế bên dưới áp dụng giải thuật lặp tuần tự (Iterative pipeline) thông qua `JSONL` để trích xuất dần những đặc tả chỉ giới Hình Điểm (Bounding box pixel-space) và độ dài ký tự chuỗi.

In [ ]:
text_lengths = []
ignored_counts = 0
total_boxes = 0
bbox_areas = []
bbox_widths = []
bbox_heights = []

sample_boxes = []

if normalized_path.exists():
    with normalized_path.open(encoding="utf-8") as file:
        for i, line in enumerate(file):
            if not line.strip():
                continue
            obj = json.loads(line)
            total_boxes += 1

            if obj.get("ignored", False):
                ignored_counts += 1

            # Lưu giữ thông số chữ
            text_val = obj.get("text", "")
            text_lengths.append(len(text_val))

            # Lưu giữ thông số hộp giới hạn [x, y, w, h]
            bbox = obj.get("bbox", [0, 0, 0, 0])
            if len(bbox) == 4:
                w, h = bbox[2], bbox[3]
                bbox_widths.append(w)
                bbox_heights.append(h)
                bbox_areas.append(w * h)

            # Lấy 5 mẫu phân bổ đầu tiên
            if i < 5:
                sample_boxes.append(
                    {
                        "Định danh Phân mảnh (Split)": obj.get("split", "unknown"),
                        "Mã hình (Image ID)": obj.get("image_id", ""),
                        "Ngữ nghĩa (Text value)": text_val,
                        "Hộp giới hạn (Bbox)": str(bbox),
                        "Miễn trừ (Ignored)": obj.get("ignored", False),
                    }
                )

    print("✅ Quy trình giải trình tự JSONL đã hoàn tất.")
else:
    print("Cảnh báo: Đường dẫn dữ liệu OCR chuẩn hóa đang trống.")

## 2. Đo Lường Rủi Ro Sự Cố Hiển Thị (Ignored/Illegible Diagnostics)
Tỷ lệ văn bản mờ nhòe, ngoài chuẩn, hoặc bị loại trừ (Ignored properties) cần được kiểm soát minh bạch. 
Phép toán tính tỷ trọng (Proportion ratio) trực tiếp cảnh báo mức giới hạn không khả thi trong quá trình Huấn luyện (Training limit).

In [ ]:
if total_boxes > 0:
    illegible_rate = ignored_counts / total_boxes

    df_rate = pd.DataFrame(
        {
            "Hạng mục Đo lường": [
                "Tổng số Hộp văn bản (Total Boxes)",
                "Dấu tích hỏng (Ignored markings)",
                "Tỷ lệ Mù chữ (Illegible Rate)",
            ],
            "Chỉ số": [total_boxes, ignored_counts, f"{illegible_rate:.5f}"],
        }
    )
    display(df_rate)

    # Mô tả bằng bảng thống kê cho chiều dài tự phân bổ Text
    df_len = pd.DataFrame(
        text_lengths, columns=["Đầu vào từ vựng học (Character Span)"]
    )
    desc_len = df_len.describe().T
    desc_len["p95"] = np.percentile(text_lengths, 95)
    display(desc_len.round(2))
else:
    print("Chưa có thông số đo lường tỷ suất xuất hiện ngoại lệ OCR.")

## 3. Khảo Sát Đồ Thị Hộp Giới Hạn Văn Bản (BBox Graphical Survey)
Đo đạc tính phi định hình (Morphological instability) bằng các giá trị Phân vị thứ 50 (Median), Phân vị thứ 95 trong biểu đồ Mật độ Đồ thị (Boxplots Density Visualization).
Phương pháp so sánh diện tích (Area scale correlation) làm rõ mật độ bao phủ pixel.

In [ ]:
if total_boxes > 0 and bbox_widths:
    # Tính toán bảng phần trăm
    df_bbox = pd.DataFrame(
        {
            "Chiều Rộng (Width)": bbox_widths,
            "Chiều Cao (Height)": bbox_heights,
            "Diện tích Tích chập (Area)": bbox_areas,
        }
    )

    desc_bbox = df_bbox.describe(percentiles=[0.5, 0.95]).T
    display(desc_bbox[["mean", "50%", "95%"]].round(2))

    # Vẽ biểu đồ KDE thể hiện mật độ Diện Tích Hộp Giới Hạn
    plt.figure(figsize=(10, 5))
    # Khống chế 99 percentile bỏ qua nhiễu lớn khi vẽ Density Curve
    threshold = np.percentile(bbox_areas, 95)
    sns.kdeplot(
        data=df_bbox[df_bbox["Diện tích Tích chập (Area)"] < threshold],
        x="Diện tích Tích chập (Area)",
        fill=True,
        color="purple",
        alpha=0.5,
    )
    plt.title(
        "Đường Cong Phân Phối Mật Độ Diện Tích Khung Chữ (95th Percentile Filtered)"
    )
    plt.xlabel("Diện tích (Area in Pixels)")
    plt.ylabel("Mật độ Tần suất")
    plt.show()

    # Thể hiện văn bản trực tiếp
    display(pd.DataFrame(sample_boxes))
else:
    print("Chưa có dữ liệu khung bao để tạo Bounding Graph.")